# Overhead plots

The idea is to create a plot for the CPU and latency overhead when there's an Agent actually taking decisions and when there is no Agent.

The first approximation is: when there is no agent --> when the agent picks always the same value, in that case I am assuming the cost is basically 0. I am saying this because we measure the CPU consumption of the Aggregate, and if the D value does not change there is not really much the Aggregate is doing.

Of course it would be more "correct" to do experiments with an Aggregate that is not connected at all with the RL framework part, but in the interest of time we start with this.

Now, we need to go back to the actual log data and see a bit what we have... the data is here:  `data/10/linearroad-CCR/5/600`. From the data:
- we have all the D values
- we have 100 episodes for the D value
- even when D is 10, the episode runs for some 80 seconds

Hence:
- we could take the middle 60 seconds from each D and from each episode and log them
- we already have the python script that "cuts" individual episodes from the single log file, so we could add an optional parameter that asks if we want to dump the individual logs (default: no)
- then we do the same for the RL agent experiment and we have what we need to start creating the new plots

To avoid messing with the data we already have, I am creating a copy of the folder so we create the new data in the new folder. I'm copying it into `data/exp016.22.overhead/linearroad/baseline`

- `mkdir data/exp016.22.overhead`
- `mkdir data/exp016.22.overhead/linearroad`
- `mkdir data/exp016.22.overhead/linearroad/baseline`
- `cp -r data/10/linearroad-CCR/5/600/* data/exp016.22.overhead/linearroad/baseline/`

Then I am running:
- `./scripts/create_plots_for_exp.sh data/exp016.22.overhead/linearroad/baseline True 0` but changing `reward_pattern` to `"Old` to see what happens

This seems to work
Now I'm adding --dumpdata to `plot_experiment_stats_exp.py` to dump data too (if the paramater is passed)

This is how you run it:
- `./scripts/create_plots_for_exp.sh data/exp016.22.overhead/linearroad/baseline True 0 1 2 3 4 5 6 7 8 9 10`

Now doing the same for synthetic

And now trying with the agent one, again:
- copy the data
- Run the script (this time with the New not the Old parameter)
- `mkdir data/exp016.22.overhead/linearroad/agent`
- `cp -r data/exp16-5/WELAW/linear/* data/exp016.22.overhead/linearroad/agent/`
- `./scripts/create_plots_for_exp.sh data/exp016.22.overhead/linearroad True agent`
- `mkdir data/exp016.22.overhead/synthetic/agent`
- `cp -r data/exp16-5/WELAW/synthetic/* data/exp016.22.overhead/synthetic/agent/`
- `./scripts/create_plots_for_exp.sh data/exp016.22.overhead/synthetic True agent`



# 250107

- I just realized the way I wanted to prepare the plots is actually not the best, because we might be comparing executions at completely different points in times, in which CPU and Latency are different not because of the Agent, but because of the different point in time in the exectuion
- We need to run from scratch an Aggregate without nothing and one with the Agent
- Let's start running one without nothing, what we could do is that
  - can we define a single episode and a single step but make the inter-step period long enough?
  - can we force the random seed to be a specific value to make sure the experiment is always starting at the same point?
  - We should also think about the policy barrier, if that plays a role

So now:
- I can run a single episode of the length I want with basically no action taking place
- I can choose the random seed and the state measurement check period
- The next would be
  - have a random that takes random steps but only increase/decrease/stay --> Actually this is already what is happenning it seems, will doublecheck but it should be like that
  - setup a new experiment in which there is still 1 episode but a certain number of steps
  - and here I would use the WELAW policy by the way
  - then we should have the data for one repetition (NOTE: one episode per run, since it will be difficult to align episodes across runs if one does only one step and the other an arbitrary number of steps)
  - note also that if X is the state monitoring check period the experiment will be 2X long with an action at X, so we should take the data during the first X I think

Now I can:
- reuse the script from yesterday to extract the data (putting Old in the internal parameter of create_plots_for_exp)
- `./scripts/create_plots_for_exp.sh data/overhead/linearroad/5/600 True 10 r`

- From the results it seems to work, but the experiment length for the "no-agent case" is strange...
- It might be because of the time it takes in between the reset and the time the SPE is actually ready... Maybe 2 actions just to double check

Now I should run a couple of experiments, before starting creating plots, so the same setup but for a bunch of random seeds I guess
- running start_all_evaluation_CCR.sh
- `./scripts/create_plots_for_exp.sh data/overhead/linearroad True 10/2 10/5 10/122 10/123 10/1242 r/2 r/5 r/122 r/123 r/1242`

Now creating a python script to merge the data

Write a python script that using argsparse gets:
- a folder A, a folder B, a series of experiments ids, an output csv file data
- for each id, reads the files A/id/eps/CPU-agg.average.000.csv and B/id/eps/CPU-agg.average.000.csv (there are 2 columns, timestamp and value, with headers) and puts the column i and the column timestamp in the output csv file and the value column of each file as extra columns in the output csv. Note in the file name i uses 3 digits and also note that the timestamp column should contain the union of the timestamp columns found in the two file
- then it also adds the columns taken from the files A/id/eps/latency.average.000.csv and B/id/eps/latency.average.000.csv

- `python plotting/merge_overhead_data.py data/overhead/linearroad/10 data/overhead/linearroad/r 2 5 122 123 1242 data/overhead/linearroad/merged.csv`
- `python plotting/compute_overheads_from_merged_data.py data/overhead/linearroad/merged.csv data/overhead/linearroad/diffs.csv`
- `python plotting/plot_overheads.py data/overhead/linearroad/diffs.csv data/overhead/linearroad/plot.pdf`

Now doing the same with synthetic:
- `./scripts/start_all_evaluation_CCR.sh` (after updating the internal parameters to synthetic)
- `./scripts/create_plots_for_exp.sh data/overhead/synthetic True 10/2 10/5 10/122 10/123 10/1242 r/2 r/5 r/122 r/123 r/1242`
- `python plotting/merge_overhead_data.py data/overhead/synthetic/10 data/overhead/synthetic/r 2 5 122 123 1242 data/overhead/synthetic/merged.csv`
- `python plotting/compute_overheads_from_merged_data.py data/overhead/synthetic/merged.csv data/overhead/synthetic/diffs.csv`
- `python plotting/plot_overheads.py data/overhead/synthetic/diffs.csv data/overhead/synthetic/plot.pdf`

# 250107

- one question: in the process I am using all the data from each episode or only the central portion?
  - Yes, and I can see two problems as of now
    - For the synthetic use case I am not running enough steps
    - I should only take the periods in which the latency is smaller than the hard threshold and the CPU is less than 0.99 or whatever, because the episdoe would terminate there
- Now it looks better. I am also thinking we could have an extra experiment in which we keep saying 10 all the time but with small steps, that measure the overhead of the framework without changes in the compression!
  - Since it's the same D value 10 I am adding steps and state-check-period in the experiment id to avoid conflicts in the names


linear road:
- `./scripts/start_all_evaluation_CCR.sh` (after updating the internal parameters to linear road)
- `./scripts/create_plots_for_exp.sh data/overhead/linearroad/10/80/0.5 True 2 5 122 123 1242`
- `python plotting/merge_overhead_data.py data/overhead/linearroad/10 data/overhead/linearroad/10/80/0.5 2 5 122 123 1242 data/overhead/linearroad/merged_CCR.csv`
- `python plotting/compute_overheads_from_merged_data.py data/overhead/linearroad/merged_CCR.csv data/overhead/linearroad/diffs_CCR.csv`
- `python plotting/plot_overheads.py data/overhead/linearroad/diffs_CCR.csv data/overhead/linearroad/plot_CCR.pdf`

synthetic:
- `./scripts/start_all_evaluation_CCR.sh` (after updating the internal parameters to synthetic)
- `./scripts/create_plots_for_exp.sh data/overhead/synthetic/10/80/0.5 True 2 5 122 123 1242`
- `python plotting/merge_overhead_data.py data/overhead/synthetic/10 data/overhead/synthetic/10/80/0.5 2 5 122 123 1242 data/overhead/synthetic/merged_CCR.csv`
- `python plotting/compute_overheads_from_merged_data.py data/overhead/synthetic/merged_CCR.csv data/overhead/synthetic/diffs_CCR.csv`
- `python plotting/plot_overheads.py data/overhead/synthetic/diffs_CCR.csv data/overhead/synthetic/plot_CCR.pdf`



bo = base only
co = communication only
aa = active agent
lr = linear road
s = synthetic
Write a python script that using argsparse takes 4 input files bo_co_lr, bo_aa_lr, bo_co_s, bo_aa_s and one output pdf.
Each input file is a csv with columns (headers in the csv) "timestamp,exp_id,stat,value_diff,percentage_diff"
the output pdf is a plot with subplots, left and right. The left contains violin plots of value_diff for stat "CPU" and bo_co_lr, bo_aa_lr, bo_co_s, bo_aa_s. The right contains violin plots for stat "Latency" and bo_co_lr, bo_aa_lr, bo_co_s, bo_aa_s.

The experiments have been completed. Now:
- `./scripts/create_plots_for_exp.sh data/overhead/linearroad/10/80/0.5 True 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20`
- `./scripts/create_plots_for_exp.sh data/overhead/linearroad/10/2/120.0 True 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20`
- `./scripts/create_plots_for_exp.sh data/overhead/linearroad/r/40/0.5 True 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20`
- `./scripts/create_plots_for_exp.sh data/overhead/synthetic/10/80/0.5 True 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20`
- `./scripts/create_plots_for_exp.sh data/overhead/synthetic/10/2/120.0 True 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20`
- `./scripts/create_plots_for_exp.sh data/overhead/synthetic/r/40/0.5 True 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20`
- `python plotting/merge_overhead_data.py data/overhead/linearroad/10/2/120.0 data/overhead/linearroad/10/80/0.5 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 data/overhead/linearroad/merged_10.csv`
- `python plotting/merge_overhead_data.py data/overhead/linearroad/10/2/120.0 data/overhead/linearroad/r/40/0.5 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 data/overhead/linearroad/merged_r.csv`
- `python plotting/merge_overhead_data.py data/overhead/synthetic/10/2/120.0 data/overhead/synthetic/10/80/0.5 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 data/overhead/synthetic/merged_10.csv`
- `python plotting/merge_overhead_data.py data/overhead/synthetic/10/2/120.0 data/overhead/synthetic/r/40/0.5 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 data/overhead/synthetic/merged_r.csv`
- `python plotting/compute_overheads_from_merged_data.py data/overhead/linearroad/merged_10.csv data/overhead/linearroad/diffs_10.csv`
- `python plotting/compute_overheads_from_merged_data.py data/overhead/linearroad/merged_r.csv data/overhead/linearroad/diffs_r.csv`
- `python plotting/compute_overheads_from_merged_data.py data/overhead/synthetic/merged_10.csv data/overhead/synthetic/diffs_10.csv`
- `python plotting/compute_overheads_from_merged_data.py data/overhead/synthetic/merged_r.csv data/overhead/synthetic/diffs_r.csv`
- `python plotting/plot_overheads.py data/overhead/linearroad/diffs_10.csv data/overhead/linearroad/plot_10.pdf`
- `python plotting/plot_overheads.py data/overhead/linearroad/diffs_r.csv data/overhead/linearroad/plot_r.pdf`
- `python plotting/plot_overheads.py data/overhead/synthetic/diffs_10.csv data/overhead/synthetic/plot_10.pdf`
- `python plotting/plot_overheads.py data/overhead/synthetic/diffs_r.csv data/overhead/synthetic/plot_r.pdf`
- `python plotting/plot_overheads_combined.py data/overhead/linearroad/diffs_10.csv data/overhead/linearroad/diffs_r.csv data/overhead/synthetic/diffs_10.csv data/overhead/synthetic/diffs_r.csv data/overhead/plot.pdf`

- Now computing stats for CPU and memory of the agent
- Using a script called print_agent_overhead.py

250114

- Redoing the experiments to have higher CPU consumption on the baseline
- Doing it for linear road for now, using always compression 5 basically
- comparing only base and constant D, no random
- `./scripts/create_plots_for_exp.sh data/overhead2/linearroad/5/80/0.5 True 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20`
- `./scripts/create_plots_for_exp.sh data/overhead2/linearroad/5/2/120.0 True 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20`
- `./scripts/create_plots_for_exp.sh data/overhead2/synthetic/7/80/0.5 True 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20`
- `./scripts/create_plots_for_exp.sh data/overhead2/synthetic/7/2/120.0 True 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20`
- `python plotting/merge_overhead_data.py data/overhead2/linearroad/5/2/120.0 data/overhead2/linearroad/5/80/0.5 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 data/overhead2/linearroad/merged_5.csv`
- `python plotting/merge_overhead_data.py data/overhead2/synthetic/7/2/120.0 data/overhead2/synthetic/7/80/0.5 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 data/overhead2/synthetic/merged_7.csv`
- `python plotting/compute_overheads_from_merged_data.py data/overhead2/linearroad/merged_5.csv data/overhead2/linearroad/diffs_5.csv`
- `python plotting/compute_overheads_from_merged_data.py data/overhead2/synthetic/merged_7.csv data/overhead2/synthetic/diffs_7.csv`
- `python plotting/plot_overheads.py data/overhead2/linearroad/diffs_5.csv data/overhead2/linearroad/plot_5.pdf`
- `python plotting/plot_overheads.py data/overhead2/synthetic/diffs_7.csv data/overhead2/synthetic/plot_7.pdf`
- `python plotting/plot_overheads_combined.py data/overhead2/linearroad/diffs_5.csv data/overhead/linearroad/diffs_r.csv data/overhead2/synthetic/diffs_7.csv data/overhead/synthetic/diffs_r.csv data/overhead2/plot.pdf`

- `./scripts/create_plots_for_exp.sh data/overhead2/linearroad/3/80/0.5 True 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20`
- `./scripts/create_plots_for_exp.sh data/overhead2/linearroad/3/2/120.0 True 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20`
- `python plotting/merge_overhead_data.py data/overhead2/linearroad/3/2/120.0 data/overhead2/linearroad/3/80/0.5 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 data/overhead2/linearroad/merged_3.csv`
- `python plotting/compute_overheads_from_merged_data.py data/overhead2/linearroad/merged_3.csv data/overhead2/linearroad/diffs_3.csv`
- `python plotting/plot_overheads.py data/overhead2/linearroad/diffs_3.csv data/overhead2/linearroad/plot_3.pdf`
- `python plotting/plot_overheads_combined.py data/overhead2/linearroad/diffs_3.csv data/overhead/linearroad/diffs_r.csv data/overhead2/synthetic/diffs_7.csv data/overhead/synthetic/diffs_r.csv data/overhead2/plot3.pdf`